# Volcano + GSEA on a NEBULA `deg_results.csv`Simple, standalone interpretation notebook: point `RESULTS_CSV` at any `deg_results.csv`produced by `SlideTags_DEG_factors.ipynb` (works for either "Run" section - healthy vsdiseased, or cell type vs cell type), pick which model term to look at, and this makes avolcano plot + runs pre-ranked GSEA. Reuses `DEG.plot_vulcano` (`utils/DEG.py`) - same function`NucSeq_DEG_interpretation.ipynb` uses, but without that notebook's project-specific extras(custom gene sets, factor/PCA interpretation, ORA, cross-cell-type comparisons, etc.).

In [ ]:
%load_ext autoreload%autoreload 2import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport osimport refrom dotenv import load_dotenv; load_dotenv()from statsmodels.stats.multitest import multipletestsimport gseapy as gpfrom gseapy import dotplotfrom utils import DEG%matplotlib inline

In [ ]:
# Point this at any deg_results.csv produced by SlideTags_DEG_factors.ipynb, e.g.:#   .../DEG/gradient_score_nebula/Matrix/deg_results.csv                 (healthy vs diseased, cell type "Matrix")#   .../DEG/gradient_score_nebula/Matrix_vs_Patch_allConditions/deg_results.csv   (cell type vs cell type)RESULTS_CSV = "/home/gdallagl/myworkdir/XDP/data/XDP/SlideTags_dataset/DEG/gradient_score_nebula/Matrix/deg_results.csv"# The contrast term to pull out of deg_results.csv, same convention as NucSeq_DEG_interpretation.ipynb:#   column names are "p_{CONTRAST_VARIABLE}{CONTRAST_STIM}", "logFC_{CONTRAST_VARIABLE}{CONTRAST_STIM}", "se_{...}"# For a healthy-vs-diseased run: CONTRAST_VARIABLE="condition", CONTRAST_STIM="diseased"# For a cell-type-vs-cell-type run: CONTRAST_VARIABLE="ct_for_deg", CONTRAST_STIM=<the CT_STIM you used>CONTRAST_VARIABLE = "condition"CONTRAST_STIM = "diseased"GENE_COL = "gene"INTERCEPT_COL = "logFC_(Intercept)"# significance / ranking parametersLOGFC_THR = 0.5     # |log2FC| threshold for the volcano plotPVAL_THR = 0.05      # threshold on BH-adjusted p-value, both for the volcano plot and for calling a gene a DEGMETHOD_MULTIPLE_TEST = "fdr_bh"REMOVE_NOISY_GENES_FROM_GSEA = True     # drop ribosomal/mito/lincRNA/miRNA/unmapped-ENSG genes from the GSEA rankingREMOVE_EXTREME_INTERCEPT_GENES = True   # drop genes with intercept logFC < -20 (near-unexpressed at baseline -> unreliable logFC)THR_FDR_GSEA = 0.05

# Load results

In [ ]:
df = pd.read_csv(RESULTS_CSV)print(f"{len(df)} genes in {RESULTS_CSV}")PVAL_COL  = f"p_{CONTRAST_VARIABLE}{CONTRAST_STIM}"LOGFC_COL = f"logFC_{CONTRAST_VARIABLE}{CONTRAST_STIM}"SE_COL    = f"se_{CONTRAST_VARIABLE}{CONTRAST_STIM}"available_terms = sorted(c[len("p_"):] for c in df.columns if c.startswith("p_"))print("Available contrast terms in this file (p_<term>):", available_terms)assert PVAL_COL in df.columns, f"{PVAL_COL!r} not found - check CONTRAST_VARIABLE/CONTRAST_STIM against the list above"# Wald z-score on the natural-log scale NEBULA fits on (compute BEFORE rescaling logFC below)df["z_score"] = df[LOGFC_COL] / df[SE_COL]# NEBULA's logFC is on the natural-log scale (standard for NB/Poisson GLMs) -> convert to log2# for the usual "fold change" reading used in the volcano plot / GSEA directiondf[LOGFC_COL] = df[LOGFC_COL] / np.log(2)# BH-FDR across genes tested for this specific termCOL_P_ADJ = f"adj_{PVAL_COL}"mask = df[PVAL_COL].notna()df.loc[mask, COL_P_ADJ] = multipletests(df.loc[mask, PVAL_COL], method=METHOD_MULTIPLE_TEST)[1]n_sig = ((df[COL_P_ADJ] <= PVAL_THR) & (df[LOGFC_COL].abs() >= LOGFC_THR)).sum()print(f"{n_sig} / {mask.sum()} genes significant at FDR<={PVAL_THR} & |log2FC|>={LOGFC_THR}")df.sort_values(COL_P_ADJ).head(15)[[GENE_COL, LOGFC_COL, PVAL_COL, COL_P_ADJ]]

# Volcano plot

In [ ]:
OUT_DIR = os.path.join(os.path.dirname(RESULTS_CSV), "interpretation")os.makedirs(OUT_DIR, exist_ok=True)fig, ax = plt.subplots(figsize=(9, 7))DEG.plot_vulcano(    df,    logfc_col=LOGFC_COL,    pval_col=COL_P_ADJ,    gene_col=GENE_COL,    pval_thresh=PVAL_THR,    logfc_thresh=LOGFC_THR,    to_label=15,    ax=ax,)ax.set_title(f"{CONTRAST_VARIABLE} = {CONTRAST_STIM}")fig.savefig(f"{OUT_DIR}/volcano_{CONTRAST_VARIABLE}{CONTRAST_STIM}.png", dpi=150, bbox_inches="tight")plt.show()

# GSEA (pre-ranked)Ranks genes by `sign(log2FC) * -log10(raw p-value)` (GSEA does its own permutation-basedsignificance testing, so the *raw*, not FDR-adjusted, p-value is used for ranking magnitude -the adjusted p-value above is only for calling significance on the volcano plot).

In [ ]:
def filter_noisy_genes(ranked_series, verbose=True):    """Drop common non-informative snRNA-seq gene classes (ribosomal, mito, lincRNA, snoRNA,    miRNA, unmapped Ensembl IDs) from a GSEA ranking - these are rarely biologically    interpretable hits and can dominate gene-set overlaps by sheer count."""    pattern = re.compile(r'''^(        RPL   | RPS   | MRPL  | MRPS  |        MT-                            |        LINC                           |        SNHG  | SNORA | SNORD          |        MIR[0-9]                       |        RNU                            |        ENSG    )''', re.IGNORECASE | re.VERBOSE)    noisy = [g for g in ranked_series.index if pattern.match(str(g))]    n_before = len(ranked_series)    ranked_series = ranked_series.drop(index=noisy)    if verbose:        print(f"filter_noisy_genes: removed {n_before - len(ranked_series)} genes ({n_before} -> {len(ranked_series)} remaining)")    return ranked_seriesdf_rank = df.copy()if REMOVE_EXTREME_INTERCEPT_GENES:    extreme = df_rank[INTERCEPT_COL] < -20    print(f"Removed {extreme.sum()} genes with extreme intercept logFC (< -20, near-unexpressed at baseline)")    df_rank = df_rank[~extreme]df_rank = df_rank.dropna(subset=[PVAL_COL, LOGFC_COL])df_rank["rank_metric"] = np.sign(df_rank[LOGFC_COL]) * -np.log10(df_rank[PVAL_COL].clip(lower=1e-300))df_rank = df_rank.sort_values("rank_metric", ascending=False)ranked_series = df_rank.set_index(GENE_COL)["rank_metric"]if REMOVE_NOISY_GENES_FROM_GSEA:    ranked_series = filter_noisy_genes(ranked_series)sns.histplot(ranked_series, bins=100)plt.xlabel("rank metric"); plt.show()print(f"{len(ranked_series)} genes ranked (top: {ranked_series.index[0]} {ranked_series.iloc[0]:+.2f}, "      f"bottom: {ranked_series.index[-1]} {ranked_series.iloc[-1]:+.2f})")

In [ ]:
# A short, standard set of libraries - add/remove as needed (each is FDR-corrected only# against itself, so don't compare FDR values directly across libraries)gsea_libraries = [    ("MSigDB_Hallmark_2020",       15, 250),    ("GO_Biological_Process_2025", 15, 250),    ("Reactome_Pathways_2024",     15, 250),    ("SynGO_2024",                 15, 250),]gsea_sig_results = {}for db_name, min_size, max_size in gsea_libraries:    print(f"\n{'─'*50}\nGSEA  ▶  {db_name}\n{'─'*50}")    pre_res = gp.prerank(        rnk=ranked_series,        gene_sets=db_name,        seed=42,        permutation_num=1000,        min_size=min_size,        max_size=max_size,        verbose=False,        threads=8,    )    df_gsea = pre_res.res2d.sort_values("FDR q-val")    df_gsea.to_csv(f"{OUT_DIR}/gsea_{db_name}_all.csv", index=False)    df_gsea_sig = df_gsea[df_gsea["FDR q-val"] <= THR_FDR_GSEA].copy()    df_gsea_sig.to_csv(f"{OUT_DIR}/gsea_{db_name}_significant.csv", index=False)    gsea_sig_results[db_name] = df_gsea_sig    print(f"{len(df_gsea)} terms tested -> {len(df_gsea_sig)} significant (FDR<={THR_FDR_GSEA})")    if len(df_gsea_sig):        display(df_gsea_sig[["Term", "NES", "FDR q-val", "Lead_genes"]].head(15))

# Dotplot (per library, top significant terms)

In [ ]:
for db_name, df_sig in gsea_sig_results.items():    if len(df_sig) == 0:        print(f"{db_name}: nothing significant, skipping plot")        continue    res2d_plot = df_sig.copy()    res2d_plot["FDR q-val"] = res2d_plot["FDR q-val"].replace(0, 1e-300)  # avoid colormap crash when FDR=0    ax = dotplot(        res2d_plot,        column="FDR q-val",        title=db_name,        cutoff=THR_FDR_GSEA,        top_term=15,        figsize=(6, 6),    )    plt.savefig(f"{OUT_DIR}/gsea_{db_name}_dotplot.png", dpi=150, bbox_inches="tight")    plt.show()